# Saliency incremental threshold calibration

01의 사전 지정 FIQA 계열 기준 방법에 saliency 특징이 추가 정보를 제공하는지 검증합니다. **첫 실행은 readiness/faithfulness 확인만 수행합니다.** Gate가 blocked이면 코드 오류가 아니라 현재 입력으로 보정 실험을 승인할 수 없다는 뜻입니다.

Saliency의 1차 역할(압축과 인식 근거·검색 행동 분석)은 유지합니다. 이 노트북은 causal/deployment 성능이나 FPIR의 이론적 보장을 주장하지 않습니다.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'research').is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SURVFACE_RUN_CANDIDATES = {
    'arcface': PROJECT_ROOT / 'runs/survface_20260902/20260902-R001-61915edf_step4_survface_arcface-7972a704552df378345f',
    'adaface': PROJECT_ROOT / 'runs/survface_20260830/20260830-R001-ec6e5d4a_step4_survface_adaface-4df25b75e065b0b9ed43',
    'magface': PROJECT_ROOT / 'runs/survface_20260831/20260831-R001-6695386d_step4_survface_magface-6931178ad2025e1b3799',
    'edgeface': PROJECT_ROOT / 'runs/survface_20260901/20260901-R001-56c2f3ed_step4_survface_edgeface-a348c305af33c223b337',
}
SOURCE_MODEL = 'arcface'
FIQA_VARIANT = 'L'
COMPRESSION_PROFILE = 'pq_512_m128_b8'
SEARCH_MODE = 'pq_adc_exhaustive'
METRIC_CONTRACT = 'genuine-score-topk-v2'
RESULT_ROOT = PROJECT_ROOT / 'results/calibration'
# test 결과를 보고 최선 방법을 자동 선택하지 않음. 01과 같은 hyperparameter를 사전 지정.
BASELINE_METHOD = 'continuous_fiqa'  # 또는 continuous_fiqa_margin / continuous_fiqa_margin_distortion
RETRIEVAL_FEATURE_DIR = None       # retrieval 기준 방법이면 01의 완료 feature 경로 필수
SALIENCY_FEATURE_DIR = None        # None: 현재 run의 population saliency(진단용)
FAITHFULNESS_DIR = None            # None: 해당 run의 명시적 faithfulness_v2_all(진단용)
PARTITION_SEED = 8972
TARGET_FPIRS = (.01, .05, .10, .20, .30)
SAFETY_FRACTION = .30
KNOT_QUANTILES = (1/3, 2/3)
SMOOTHING = .01
RIDGE = .001
MAX_ITERATIONS = 2000
MARGIN_SLOPE_CAP = .95
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 8972
RUN_INCREMENTAL_CALIBRATION = False
WRITE_INCREMENTAL_RESULTS = False
RUN_SPLIT_STABILITY = False
WRITE_SPLIT_RESULTS = False
SPLIT_SEEDS = (*range(19), 8972)
if SOURCE_MODEL not in SURVFACE_RUN_CANDIDATES or FIQA_VARIANT not in ('S', 'L'):
    raise ValueError('명시된 FR 모델과 FIQA variant를 선택하세요.')
if WRITE_INCREMENTAL_RESULTS and not RUN_INCREMENTAL_CALIBRATION:
    raise ValueError('WRITE requires RUN_INCREMENTAL_CALIBRATION')
if WRITE_SPLIT_RESULTS and not RUN_SPLIT_STABILITY:
    raise ValueError('WRITE requires RUN_SPLIT_STABILITY')
if RUN_SPLIT_STABILITY and (len(SPLIT_SEEDS) < 2 or len(set(SPLIT_SEEDS)) != len(SPLIT_SEEDS)):
    raise ValueError('분할 안정성에는 중복 없는 seed 두 개 이상이 필요합니다.')


## 1. 방법과 gate

- 기준 방법은 01에서 사전 지정한 Continuous FIQA 계열입니다. 같은 설정으로 calibration에서 재적합하므로 01 결과 파일이 자동 연결되거나 test 우승 방법이 선택되지 않습니다.
- 추가 조건: +outside_face_attention(얼굴 밖 주의량), +saliency_entropy(분포 엔트로피), +두 특징. 모든 스케일·knot·회귀 적합은 calibration fit non-mated에서만, 안전 보정은 held-out safety에서만 수행합니다.
- **High−Low와 High−Random의 identity-cluster 95% CI 하한이 모두 양수**여야 strong faithfulness입니다. Random은 가림 실험의 음성 대조군이지 threshold 변수나 랜덤 보정 계수가 아닙니다.
- 같은 query cohort를 유지하려고 02는 calibration/test 모두 100% 유효 coverage 및 alignment hash를 요구합니다. 누락 행 제거·0 대치로 결과를 만들지 않습니다.
- split별 gallery를 사용해 생성한 label-free origin-top1 saliency여야 합니다. 원본 gallery 접근이 필요하므로 압축 DB만으로 배포 가능한 방법이라고 주장하지 않습니다.
- faithfulness가 test와 겹치면 그 결과로 사용 여부를 결정하지 않습니다. 기존 test faithfulness는 설명용으로 표시하되 실행 gate 근거로는 불충분합니다.


In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown
from research.runtime.hashing import sha256_file
from research.experiments.fiqa_threshold_calibration import load_condition_score_artifact
from research.experiments.fiqa_retrieval_features import load_retrieval_features
from research.experiments.saliency_incremental_calibration import (
    load_saliency_incremental_inputs, assess_incremental_gate,
    run_saliency_incremental_calibration, write_saliency_incremental_result,
)
from research.fiqa import CRFIQA_VARIANTS, load_fiqa_score_artifact

source_dir = SURVFACE_RUN_CANDIDATES[SOURCE_MODEL]
source = json.loads((source_dir / 'run_manifest.json').read_text(encoding='utf8'))
if source.get('status') != 'completed' or not (source_dir / 'COMPLETED').is_file():
    raise ValueError('완료 source run이 필요합니다.')
run_id = source['run_id']
condition_dir = RESULT_ROOT / 'condition_scores' / run_id / f'{COMPRESSION_PROFILE}__{SEARCH_MODE}' / METRIC_CONTRACT
condition = load_condition_score_artifact(condition_dir)
for key, expected in {
    'source_run_id': run_id, 'model_uid': source['config']['model_uid'],
    'compression_profile': COMPRESSION_PROFILE, 'search_mode': SEARCH_MODE,
    'metric_contract': METRIC_CONTRACT, 'source_run_manifest_sha256': sha256_file(source_dir / 'run_manifest.json'),
}.items():
    if condition.manifest.get(key) != expected:
        raise ValueError(f'condition lineage 불일치: {key}')
saliency_dir = Path(SALIENCY_FEATURE_DIR) if SALIENCY_FEATURE_DIR is not None else source_dir / 'artifacts/step2_workflow/saliency_population'
faithfulness_dir = Path(FAITHFULNESS_DIR) if FAITHFULNESS_DIR is not None else PROJECT_ROOT / 'results/paper/survface' / run_id / 'faithfulness_v2_all'
inputs = load_saliency_incremental_inputs(condition, saliency_dir, faithfulness_dir)
gate = assess_incremental_gate(condition, inputs)
display(pd.DataFrame([{
    'source_run': run_id, 'saliency_directory': str(saliency_dir),
    'faithfulness_directory': str(faithfulness_dir), 'baseline': BASELINE_METHOD,
}]))


## 2. Readiness와 High/Low/Random 결과

coverage는 동일 condition의 query ID 기준입니다. High/Low/Random은 중요한/덜 중요한/무작위 영상 영역을 가렸을 때 score가 감소한 양입니다. 아래 표의 좋은 평균 하나만으로 gate를 통과시키지 않습니다. gate 사유를 먼저 읽으세요.


In [ ]:
display(pd.DataFrame([{k: v for k, v in gate.items() if k not in ('faithfulness', 'readiness', 'reasons')}]))
display(inputs['faithfulness_summary'].loc[inputs['faithfulness_summary'].group.eq('all'),
    ['metric', 'sample_count', 'mean', 'mean_ci_lower', 'mean_ci_upper']])
display(pd.DataFrame([gate['faithfulness']]))
if gate['reasons']:
    display(pd.DataFrame({'blocked_reason': gate['reasons']}))
else:
    display(Markdown('Gate 통과: 별도 RUN 설정을 켠 경우에만 적합합니다.'))


## 3. FIQA 대비 saliency 추가 비교

Gate를 통과한 후 첫 셀의 RUN_INCREMENTAL_CALIBRATION=True, 저장하려면 WRITE_INCREMENTAL_RESULTS=True로 설정하고 Kernel Restart → Run All 하세요. 01에서 margin/PQ distortion 기준을 선택했다면 RETRIEVAL_FEATURE_DIR에 동일 condition의 완료 경로를 지정해야 합니다. 이 노트북은 feature/Grad-CAM 추론을 자동 실행하지 않습니다.

FPIR은 query 단위, TPIR20은 genuine score threshold 통과 AND rank≤20이며 mated identity cluster CI를 사용합니다. 동일 target도 실제 FPIR은 다를 수 있으므로 candidate_target_met와 실제 FPIR을 반드시 같이 읽으세요. 모든 rate는 0~1, 차이를 ×100하면 %p입니다.


In [ ]:
comparison = stability = None
comparison_path = stability_path = None
fiqa = retrieval = None
if (RUN_INCREMENTAL_CALIBRATION or RUN_SPLIT_STABILITY) and gate['comparison_enabled']:
    fiqa = load_fiqa_score_artifact(RESULT_ROOT / 'fiqa_scores/survface' / CRFIQA_VARIANTS[FIQA_VARIANT].model_uid)
    if BASELINE_METHOD != 'continuous_fiqa':
        if RETRIEVAL_FEATURE_DIR is None:
            raise ValueError('01의 완료 retrieval feature 경로가 필요합니다.')
        retrieval = load_retrieval_features(RETRIEVAL_FEATURE_DIR, condition)
    settings = dict(
        baseline_method=BASELINE_METHOD, retrieval=retrieval, target_fpirs=TARGET_FPIRS,
        safety_fraction=SAFETY_FRACTION, knot_quantiles=KNOT_QUANTILES, smoothing=SMOOTHING,
        ridge=RIDGE, max_iterations=MAX_ITERATIONS, margin_slope_cap=MARGIN_SLOPE_CAP,
        resamples=BOOTSTRAP_RESAMPLES, bootstrap_seed=BOOTSTRAP_SEED,
    )
    if RUN_INCREMENTAL_CALIBRATION:
        comparison = run_saliency_incremental_calibration(condition, fiqa, inputs, partition_seeds=(PARTITION_SEED,), **settings)
        if WRITE_INCREMENTAL_RESULTS:
            comparison_path = write_saliency_incremental_result(RESULT_ROOT / 'saliency_incremental' / run_id, comparison)
        display(comparison['method_summary'][[
            'target_fpir', 'method', 'realized_fpir', 'fpir_wilson95_low', 'fpir_wilson95_high',
            'tpir_at_rank_k', 'tpir_cluster95_low', 'tpir_cluster95_high', 'target_met_on_test']])
elif not gate['comparison_enabled']:
    display(Markdown('**보정 차단:** baseline/+Saliency 적합과 성능 결과 저장을 수행하지 않았습니다.'))
else:
    display(Markdown('RUN_INCREMENTAL_CALIBRATION=False: readiness 진단만 수행했습니다.'))


In [ ]:
if comparison is not None:
    display(comparison['paired_comparisons'])
    display(Markdown(f'결과 경로: {comparison_path or "메모리 내 계산만 수행"}'))


## 4. 선택적 분할 안정성

RUN_SPLIT_STABILITY=True이면 동일 test/gallery에서 calibration fit/safety만 다시 나눕니다. 미리 정한 seed 전체를 사용하며 최적 seed를 선택하지 않습니다. 최소·중앙·최대는 기술통계이며 독립 실험 반복이나 CI가 아닙니다. Gate가 차단된 상태에서는 이 단계도 실행하지 않습니다.


In [ ]:
if RUN_SPLIT_STABILITY and gate['comparison_enabled']:
    stability = run_saliency_incremental_calibration(condition, fiqa, inputs, partition_seeds=SPLIT_SEEDS, **settings)
    if WRITE_SPLIT_RESULTS:
        stability_path = write_saliency_incremental_result(RESULT_ROOT / 'saliency_incremental_split_stability' / run_id, stability)
    display(stability['split_summary'])


## 5. 차단 시 필요한 자료와 해석 경계

1. test Grad-CAM만 있는 경우 **calibration query의 split-matched gallery 대상 saliency**를 별도로 생성해야 합니다. 기존 test 값으로 채우거나 기존 manifest를 수정하지 마세요.
2. 새로운 saliency artifact는 saliency_calibration_features/schema1/status=completed, condition_manifest_sha256, gallery_contract=split_matched_origin_top1, dataset/model/extraction/origin UID, saliency_spec_uid, target_name, saliency_features(path/bytes/sha256) 기록이 필요합니다. CSV에는 sample_id, split, aligned_content_sha256, saliency_target_name, heatmap_available, gradcam_valid_heatmap, 두 특징을 담습니다.
3. faithfulness는 동일 saliency spec의 기존 v2 High/Low/Random 형식을 사용하되 evaluation_split=calibration 또는 development, 실제 sample ID의 test 비중복, 양쪽 gain의 양의 identity-cluster 95% CI가 필요합니다. manifest 문구만 바꾸는 것은 새 검증이 아닙니다.
4. 현재 구현은 위 자료를 검증·소비하는 단계이며 **부족한 Grad-CAM/가림 실험을 생성하는 기능은 포함하지 않습니다.** 생성은 GPU·시간 범위를 별도로 정해 수행해야 합니다.

산출물(실행 성공 시): method_summary.csv, paired_comparisons.csv, models.csv, split_summary.csv, manifest.json. 입력/구현 hash 기반 새 UID에 저장하며 기존 결과는 덮어쓰지 않습니다. 다중 비교·calibration uncertainty·추론 비용을 모두 확증한 것은 아닙니다. 개선되지 않는 결과도 그대로 보고하며 공통 보고서에 자동 승격하지 않습니다.


In [ ]:
display(pd.DataFrame([
    {'stage': 'readiness / faithfulness', 'status': gate['status'], 'artifact': None},
    {'stage': 'single split', 'status': 'computed' if comparison is not None else ('blocked' if not gate['comparison_enabled'] else 'not_run'), 'artifact': str(comparison_path) if comparison_path else None},
    {'stage': 'split stability', 'status': 'computed' if stability is not None else ('blocked' if not gate['comparison_enabled'] else 'not_run'), 'artifact': str(stability_path) if stability_path else None},
]))
